# Basic Investment Model — SMM Validation

This notebook validates the Simulated Method of Moments (SMM) implementation using the frictionless basic-investment model, where the analytical policy is exact (`cost_convex == 0`, `cost_fixed == 0`). Any errors must be in SMM, not model solution.

**Two-step estimation procedure:**

- **Stage 1:** Minimize Q(beta) with W = I using `dual_annealing` (global search) + Powell refinement.
- **Stage 2:** Warm-start from beta_hat_1 with W = Omega_hat^{-1}, local Powell only.

**Tables produced:**

1. Moment fit — target vs fitted moments at beta_hat
2. Parameter estimates — true, estimate, SE, t-test, J-test
3. Monte Carlo summary — bias, RMSE, SD, SE across J replications

In [1]:
from pathlib import Path
import os
import sys
import time
import warnings
from dataclasses import asdict

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

warnings.filterwarnings(
    'ignore',
    message=r'.*does not produce the same series as CPU implementation.*',
)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display

from src.v2.estimation import (
    SMMMonteCarloConfig,
    SMMRunConfig,
    solve_smm,
    validate_smm,
)
from src.v2.evaluation import (
    prepare_evaluation_run,
    save_manifest_sections,
    save_summary_rows,
)
from src.v2.environments.basic_investment import (
    BasicInvestmentEnv,
    EconomicParams,
    ShockParams,
)
from src.v2.utils.seeding import fold_in_seed

np.set_printoptions(precision=6, suppress=True)

# Human-readable labels for moment and parameter names.
MOMENT_LABELS = {
    'mean_investment_assets': 'Mean I/k',
    'var_investment_assets': 'Var I/k',
    'serial_corr_investment': 'Serial Corr I/k',
    'income_ar1_beta': 'AR(1) beta',
    'income_ar1_resid_std': 'AR(1) resid std',
}
PARAM_LABELS = {
    'alpha': 'alpha',
    'rho': 'rho',
    'sigma_epsilon': 'sigma',
    'psi1': 'psi1',
}

/Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## Config

Stage 1 uses `dual_annealing` (global) + Powell refinement. Stage 2 warm-starts from the stage-1 estimate with Powell only. `optimizer_maxiter` controls the global search budget in stage 1; stage 2 is local and converges via Powell's internal criteria.

Set `SAVE_RUN = True` and a stable `RUN_TAG` to persist tables to disk for inclusion in the paper.

In [2]:
EXPERIMENT_NAME = 'basic_investment_smm_validation'
RESULTS_ROOT   = str(REPO_ROOT / 'outputs' / 'notebooks')

SAVE_RUN = True
RUN_TAG  = 'smm-validation'

PROFILE = 'MC_VALIDATION'  # Options: 'SMOKE', 'MC_VALIDATION'

PROFILES = {
    'SMOKE': {
        'description': 'Quick smoke test with small panels and few MC replications.',
        'smm': {
            'n_firms': 100,
            'horizon': 25,
            'burn_in': 75,
            'n_sim_panels': 8,
            'global_method': 'differential_evolution',
            'optimizer_maxiter': 5,
            'master_seed': (20, 26),
        },
        'mc': {
            'n_replications': 5,
        },
    },
    'MC_VALIDATION': {
        'description': 'Full Monte Carlo validation with larger panels.',
        'smm': {
            'n_firms': 500,
            'horizon': 25,
            'burn_in': 75,
            'n_sim_panels': 50,
            'global_method': 'differential_evolution',
            'optimizer_maxiter': 50,
            'master_seed': (20, 26),
        },
        'mc': {
            'n_replications': 10,
        },
    },
}

assert PROFILE in PROFILES, f'Unknown profile: {PROFILE!r}'
PROFILE_CONFIG = PROFILES[PROFILE]

# Fixed at calibrated values — not estimated by SMM.
CALIBRATED = dict(
    interest_rate=0.04,
    depreciation_rate=0.15,
)

# Ground truth for the 3 parameters estimated by SMM.
# These are unknown in real data; here we set them for MC validation.
TRUE_PARAMS = dict(
    production_elasticity=0.6,   # alpha
    rho=0.70,
    sigma=0.15,
)

ECON_PARAMS = EconomicParams(
    **CALIBRATED,
    production_elasticity=TRUE_PARAMS['production_elasticity'],
    cost_convex=0.0,
    cost_fixed=0.0,
)
SHOCK_PARAMS = ShockParams(mu=0.0, rho=TRUE_PARAMS['rho'], sigma=TRUE_PARAMS['sigma'])

# SMM estimation settings
BETA_INITIAL_GUESS = None
BETA_BOUNDS = (
    (0.10, 0.95),
    (0.0, 0.95),
    (0.01, 0.95),
)

SMM_RUN_CONFIG = SMMRunConfig(**PROFILE_CONFIG['smm'])
MC_CONFIG = SMMMonteCarloConfig(**PROFILE_CONFIG['mc'])

RUN = prepare_evaluation_run(
    EXPERIMENT_NAME,
    save_run=SAVE_RUN,
    run_tag=RUN_TAG,
    results_root=RESULTS_ROOT,
)

print(f'Profile: {PROFILE} — {PROFILE_CONFIG["description"]}')
print(f'Output: {RUN["run_dir"]}')
print(f'SMM: n_firms={SMM_RUN_CONFIG.n_firms}, horizon={SMM_RUN_CONFIG.horizon}, '
      f'burn_in={SMM_RUN_CONFIG.burn_in}, S={SMM_RUN_CONFIG.n_sim_panels}, '
      f'optimizer={SMM_RUN_CONFIG.global_method}, maxiter={SMM_RUN_CONFIG.optimizer_maxiter}')
print(f'MC:  n_replications={MC_CONFIG.n_replications}')

Profile: MC_VALIDATION — Full Monte Carlo validation with larger panels.
Output: /Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/outputs/notebooks/basic_investment_smm_validation/smm-validation
SMM: n_firms=500, horizon=25, burn_in=75, S=50, optimizer=differential_evolution, maxiter=50
MC:  n_replications=10


## Environment and Target Moments

Set up the frictionless environment (analytical policy, no solver), then generate the fake-real target moments that play the role of observed data.

In [3]:
env = BasicInvestmentEnv(econ_params=ECON_PARAMS, shock_params=SHOCK_PARAMS)
beta_true = env.smm_true_beta()
spec = env.make_smm_spec(initial_guess=BETA_INITIAL_GUESS, bounds=BETA_BOUNDS)

# Parameter overview.
beta_table = pd.DataFrame(
    {
        'parameter': [PARAM_LABELS[p] for p in spec.parameter_names],
        'true': beta_true,
        'initial_guess': spec.initial_guess,
        'lower': [b[0] for b in spec.bounds],
        'upper': [b[1] for b in spec.bounds],
    }
)
display(beta_table)
print(f'Moments: {[MOMENT_LABELS[m] for m in spec.moment_names]}')

# Generate fake-real target moments.
target_seed = fold_in_seed(SMM_RUN_CONFIG.master_seed, 'smm', 'basic_investment', 'target')
target = env.simulate_smm_target_moments(beta_true, SMM_RUN_CONFIG, target_seed)

save_manifest_sections(
    RUN,
    setup={'profile': PROFILE, 'description': PROFILE_CONFIG['description']},
    environment={
        'economic_params': asdict(ECON_PARAMS),
        'shock_params': asdict(SHOCK_PARAMS),
    },
    smm_run=asdict(SMM_RUN_CONFIG),
    monte_carlo=asdict(MC_CONFIG),
)

,parameter,true,initial_guess,lower,upper
0,alpha,0.60,0.525,0.10,0.95
1,rho,0.70,0.475,0.00,0.95
2,sigma,0.15,0.480,0.01,0.95


Moments: ['Mean I/k', 'Var I/k', 'Serial Corr I/k', 'AR(1) resid std']


PosixPath('/Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/outputs/notebooks/basic_investment_smm_validation/smm-validation/manifest.json')

## Structural Two-Step SMM

Run the full two-step SMM procedure. Stage 1 uses W = I (global search). Stage 2 warm-starts from the stage-1 estimate with W = Omega_hat^{-1} (local Powell only).

**Table 1** reports the moment fit: target moments vs simulated moments at the final estimate.
**Table 2** reports parameter estimates with standard errors and t-tests for H0: beta_hat = beta_true.

In [4]:
start = time.perf_counter()
smm_result = solve_smm(spec, target, config=SMM_RUN_CONFIG)
smm_wall_time = time.perf_counter() - start

# --- Table 1: Moment Fit ---
moment_fit = pd.DataFrame(
    {
        'moment': [MOMENT_LABELS[m] for m in spec.moment_names],
        'target': smm_result.target_moments,
        'fitted': smm_result.stage2.average_moments,
    }
)
print('Table 1. Moment Fit')
display(moment_fit)

# --- Table 2: Parameter Estimates ---
beta_hat = smm_result.beta_hat
se = smm_result.standard_errors
t_stat = (beta_hat - beta_true) / np.where(se > 0, se, np.nan)
p_value = 2.0 * (1.0 - norm.cdf(np.abs(t_stat)))

param_est = pd.DataFrame(
    {
        'parameter': [PARAM_LABELS[p] for p in spec.parameter_names],
        'true': beta_true,
        'initial_guess': spec.initial_guess,
        'estimate': beta_hat,
        'se': se,
        't_stat': t_stat,
        'p_value': p_value,
    }
)
print('Table 2. Parameter Estimates')
display(param_est)

# --- Estimation info (attached metadata) ---
est_info = [
    {'key': 'j_statistic', 'value': smm_result.j_statistic},
    {'key': 'j_p_value', 'value': smm_result.j_p_value},
    {'key': 'j_df', 'value': smm_result.j_df},
    {'key': 'stage1_optimizer', 'value': SMM_RUN_CONFIG.global_method},
    {'key': 'stage1_nit', 'value': smm_result.stage1.optimizer_nit},
    {'key': 'stage1_nfev', 'value': smm_result.stage1.optimizer_nfev},
    {'key': 'stage2_optimizer', 'value': 'Powell'},
    {'key': 'stage2_nit', 'value': smm_result.stage2.optimizer_nit},
    {'key': 'stage2_nfev', 'value': smm_result.stage2.optimizer_nfev},
    {'key': 'wall_time_sec', 'value': round(smm_wall_time, 2)},
]
print(f'\nJ = {smm_result.j_statistic:.4f}, p = {smm_result.j_p_value:.4f}, df = {smm_result.j_df}')
print(f'Stage 1: {SMM_RUN_CONFIG.global_method} + Powell | Num of Model Solves = {smm_result.stage1.optimizer_nfev}')
print(f'Stage 2: Powell (local only) | Num of Model Solves = {smm_result.stage2.optimizer_nfev}')
print(f'Wall time: {smm_wall_time:.1f}s')

# --- Export ---
save_summary_rows(RUN, moment_fit.to_dict('records'), filename='moment_fit.csv')
save_summary_rows(RUN, param_est.to_dict('records'), filename='parameter_estimates.csv')
save_summary_rows(RUN, est_info, filename='estimation_info.csv')

Table 1. Moment Fit


,moment,target,fitted
0,Mean I/k,0.192684,0.191633
1,Var I/k,0.092450,0.092624
2,Serial Corr I/k,-0.142696,-0.143236
3,AR(1) resid std,0.212351,0.212278


Table 2. Parameter Estimates


,parameter,true,initial_guess,estimate,se,t_stat,p_value
0,alpha,0.60,0.525,0.600676,0.007924,0.085328,0.932001
1,rho,0.70,0.475,0.701849,0.015561,0.118844,0.905399
2,sigma,0.15,0.480,0.150187,0.000982,0.190674,0.848781



J = 0.6164, p = 0.4324, df = 1
Stage 1: differential_evolution + Powell | Num of Model Solves = 2329
Stage 2: Powell (local only) | Num of Model Solves = 157
Wall time: 378.8s


PosixPath('/Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/outputs/notebooks/basic_investment_smm_validation/smm-validation/estimation_info.csv')

## Monte Carlo Validation

Repeat the full two-step SMM on J independent fake-real datasets. **Table 3** reports bias, RMSE, SD, and average SE across replications. The J-test reject rate measures the fraction of replications where the overidentification test rejects at 5%.

In [5]:
start = time.perf_counter()
mc_result = validate_smm(spec, beta_true, run_config=SMM_RUN_CONFIG, mc_config=MC_CONFIG)
mc_wall_time = time.perf_counter() - start

# --- Table 3: Monte Carlo Summary ---
mc_summary = pd.DataFrame(
    {
        'parameter': [PARAM_LABELS[p] for p in spec.parameter_names],
        'true': mc_result.summary.beta_true,
        'mean_estimate': mc_result.summary.mean_beta_hat,
        'bias': mc_result.summary.bias,
        'rmse': mc_result.summary.rmse,
        'sd': mc_result.summary.sd,
        'avg_se': mc_result.summary.mean_standard_error,
    }
)
print('Table 3. Monte Carlo Summary')
display(mc_summary)

# --- MC info (attached metadata) ---
mc_info = [
    {'key': 'n_replications', 'value': MC_CONFIG.n_replications},
    {'key': 'stage1_optimizer', 'value': SMM_RUN_CONFIG.global_method},
    {'key': 'stage2_optimizer', 'value': 'Powell'},
    {'key': 'optimizer_maxiter', 'value': SMM_RUN_CONFIG.optimizer_maxiter},
    {'key': 'j_test_reject_rate', 'value': round(float(mc_result.summary.j_test_size), 4)},
    {'key': 'wall_time_sec', 'value': round(mc_wall_time, 2)},
]
print(f'\nJ-test reject rate (5%): {mc_result.summary.j_test_size:.2f}')
print(f'MC wall time: {mc_wall_time:.1f}s ({MC_CONFIG.n_replications} replications)')

# --- Export ---
save_summary_rows(RUN, mc_summary.to_dict('records'), filename='mc_summary.csv')
save_summary_rows(RUN, mc_info, filename='mc_info.csv')

Table 3. Monte Carlo Summary


,parameter,true,mean_estimate,bias,rmse,sd,avg_se
0,alpha,0.60,0.600261,0.000261,0.008428,0.008879,0.008453
1,rho,0.70,0.696871,-0.003129,0.016612,0.017197,0.015872
2,sigma,0.15,0.150483,0.000483,0.001548,0.001551,0.001240



J-test reject rate (5%): 0.00
MC wall time: 3742.2s (10 replications)


PosixPath('/Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/outputs/notebooks/basic_investment_smm_validation/smm-validation/mc_info.csv')

## Conclusion

A successful validation means the SMM machinery recovers true parameters from a perturbed initial guess on a model with an exact analytical policy. This isolates SMM correctness from solver error.

Solve-in-the-loop integration (Stage B with VFI/PFI/ER solvers) is validated separately.